# Test the feature-extraction workflow

Exercises every public function in `feature_extraction_workflow.extract_features` against a sample of `df_items`. Each section verifies one stage of the pipeline before running the end-to-end driver.

In [12]:
import sys
from pathlib import Path

# Make the package importable when running from this folder
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
from nltk.stem import WordNetLemmatizer

from feature_extraction_workflow import (
    build_global_filter_regex,
    build_stopwords,
    clean_text,
    ensure_cat_columns,
    extract_features,
    filter_by_cat_3,
    merge_extracted,
    parse_list_string,
    remove_global_filters,
    run_feature_extraction,
)

pd.options.display.max_colwidth = 80
pd.options.display.max_columns = None

## 1. Load data

Read a sample of items plus the schema and global-filter configs. A 50k-row sample keeps the notebook fast while still hitting most categories.

In [13]:
DATA_DIR = PROJECT_ROOT / 'data'

df_items = pd.read_csv(
    DATA_DIR / 'meta_Home_and_Kitchen_filtered.csv',
    low_memory=False,
    # nrows=50_000,
).drop_duplicates()

print(f'df_items: {len(df_items):,} rows')
df_items[['asin', 'title', 'category', 'description', 'feature']].head(3)

df_items: 1,285,392 rows


,asin,title,category,description,feature
0,0001487795,You Are Special Today Red Plate [With Red Pen],"['Home & Kitchen', 'Kitchen & Dining', 'Dining & Entertaining', 'Dinnerware'...",['It was a time honored tradition among the early American families that whe...,[]
1,0002020300,Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy,"['Home & Kitchen', 'Home Dcor', 'Candles & Holders', 'Candles']",['VICKS INHALER relieves stuffy noses helps sinus congestion breathe easy gr...,[]
2,0006564224,Artistic Churchware Communion Cup Filler: RW525,"['Home & Kitchen', 'Kitchen & Dining', 'Dining & Entertaining', 'Glassware &...","['16 oz squeeze bottle, 1 lb.']","['Religious Supply Center', 'RW-525', 'Communion Cup Filler']"


In [14]:
with open(DATA_DIR / 'master_metadata.json') as f:
    master_metadata = json.load(f)

with open(DATA_DIR / 'global_filters.json') as f:
    global_filters = json.load(f)

print(f'master_metadata categories: {len(master_metadata)}')
print(f'global_filters phrases: {len(global_filters)}')
print(f'\nExample category schema (cat_3 = {next(iter(master_metadata))!r}):')
first_cat = next(iter(master_metadata))
print(json.dumps({first_cat: master_metadata[first_cat]}, indent=2)[:500])

master_metadata categories: 69
global_filters phrases: 41

Example category schema (cat_3 = 'Bakeware'):
{
  "Bakeware": {
    "Brand": {
      "type": "dictionary",
      "values": [
        "ann clark",
        "cybrtrayd",
        "wilton",
        "fat daddio",
        "nordic ware",
        "le creuset",
        "bia cordon bleu",
        "fox run",
        "paderno world cuisine",
        "ck product",
        "villeroy & boch",
        "coppergifts",
        "first impression",
        "ateco",
        "meri meri",
        "matfer bourgeat",
        "pampered chef",
        "chicago metallic


## 2. Test individual helpers

Smoke-test each building block in isolation before chaining them.

In [15]:
# parse_list_string: stringified list -> single space-joined string
samples = [
    "['Religious Supply Center', 'RW-525', 'Communion Cup Filler']",
    "['Set of 15 Blue, Red & Black Ball Pen']",
    '[]',
    None,
    'plain string passthrough',
]
for s in samples:
    print(f'INPUT : {s!r}')
    print(f'OUTPUT: {parse_list_string(s)!r}\n')

INPUT : "['Religious Supply Center', 'RW-525', 'Communion Cup Filler']"
OUTPUT: 'Religious Supply Center RW-525 Communion Cup Filler'

INPUT : "['Set of 15 Blue, Red & Black Ball Pen']"
OUTPUT: 'Set of 15 Blue, Red & Black Ball Pen'

INPUT : '[]'
OUTPUT: ''

INPUT : None
OUTPUT: ''

INPUT : 'plain string passthrough'
OUTPUT: 'plain string passthrough'



In [16]:
# clean_text: lowercase, number-words to digits, strip noise, lemmatize, drop stopwords
stop_words = build_stopwords()
lemmatizer = WordNetLemmatizer()

samples = [
    'You Are Special Today Red Plate [With Red Pen]',
    'Five-Drawer Wooden Storage Cabinet, 12-pc Set',
    'Nikola Tesla Photo&hellip;Quotes Poster Print (12 inch X 18 inch, Rolled)',
    'Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy',
    None,
    '',
]
for s in samples:
    print(f'BEFORE: {s!r}')
    print(f'AFTER : {clean_text(s, stop_words, lemmatizer)!r}\n')

BEFORE: 'You Are Special Today Red Plate [With Red Pen]'
AFTER : 'special today red plate red pen'

BEFORE: 'Five-Drawer Wooden Storage Cabinet, 12-pc Set'
AFTER : '5-drawer wooden storage cabinet 12 pc set'

BEFORE: 'Nikola Tesla Photo&hellip;Quotes Poster Print (12 inch X 18 inch, Rolled)'
AFTER : 'nikola tesla photo hellip quote poster print 12 inch x 18 inch rolled'

BEFORE: 'Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy'
AFTER : 'vicks inhaler relief cold sinus nasal congestion allergy'

BEFORE: None
AFTER : ''

BEFORE: ''
AFTER : ''



In [17]:
# build_global_filter_regex + remove_global_filters
filter_regex = build_global_filter_regex(global_filters)

samples = [
    'great workplace poster 100% satisfaction guaranteed',
    'beautiful candle perfect gift for any occasion',
    'no filtering needed here',
]
for s in samples:
    print(f'BEFORE: {s!r}')
    print(f'AFTER : {remove_global_filters(s, filter_regex)!r}\n')

BEFORE: 'great workplace poster 100% satisfaction guaranteed'
AFTER : 'great workplace poster 100% satisfaction guaranteed'

BEFORE: 'beautiful candle perfect gift for any occasion'
AFTER : 'beautiful candle perfect  for any occasion'

BEFORE: 'no filtering needed here'
AFTER : 'no filtering needed here'



In [18]:
# ensure_cat_columns: parse stringified category list into cat_1..cat_6
df_with_cats = ensure_cat_columns(df_items)
cat_cols = [f'cat_{i+1}' for i in range(6)]

print('cat_1..cat_6 created:', all(c in df_with_cats.columns for c in cat_cols))
print(f'\ncat_3 non-null: {df_with_cats["cat_3"].notna().sum():,} / {len(df_with_cats):,}')
df_with_cats[['asin'] + cat_cols].head(5)

cat_1..cat_6 created: True

cat_3 non-null: 1,243,641 / 1,285,392


,asin,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6
0,0001487795,Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Dinnerware,Plates,Dinner Plates
1,0002020300,Home & Kitchen,Home Dcor,Candles & Holders,Candles,None,None
2,0006564224,Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Wine & Champagne Glasses,None
3,0009046461,Home & Kitchen,Bath,Bathroom Accessories,None,None,None
4,0234937912,Home & Kitchen,Home Dcor,Home Fragrance,Incense & Incense Holders,Incense,None


In [19]:
# filter_by_cat_3: keep only rows whose cat_3 has an extraction schema
df_filtered = filter_by_cat_3(df_with_cats, master_metadata)

print(f'Before filter: {len(df_with_cats):,}')
print(f'After filter : {len(df_filtered):,} ({len(df_filtered)/len(df_with_cats)*100:.1f}% kept)')
print(f'\nUnique cat_3 values represented: {df_filtered["cat_3"].nunique()} / {len(master_metadata)}')
print(f'\nTop cat_3 values:')
print(df_filtered['cat_3'].value_counts().head(10))

Before filter: 1,285,392
After filter : 1,134,566 (88.3% kept)

Unique cat_3 values represented: 69 / 69

Top cat_3 values:
cat_3
Home Dcor Accents                       158450
Dining & Entertaining                   142544
Posters & Prints                        102456
Kitchen Utensils & Gadgets               77208
Storage & Organization                   42317
Kitchen & Table Linens                   39682
Bakeware                                 38454
Decorative Pillows, Inserts & Covers     37478
Bathroom Accessories                     36675
Candles & Holders                        34266
Name: count, dtype: int64


In [20]:
# extract_features on a few cleaned titles
sample = df_filtered[['asin', 'title', 'cat_3']].dropna(subset=['title', 'cat_3']).head(5).copy()
sample['title_cleaned'] = sample['title'].apply(
    lambda t: remove_global_filters(
        clean_text(t, stop_words, lemmatizer),
        filter_regex,
    )
)
sample['features'] = sample.apply(
    lambda r: extract_features(r['title_cleaned'], r['cat_3'], master_metadata),
    axis=1,
)

for _, row in sample.iterrows():
    print(f"cat_3   : {row['cat_3']}")
    print(f"title   : {row['title'][:80]}")
    print(f"cleaned : {row['title_cleaned'][:80]}")
    print(f"features: {row['features']}")
    print()

cat_3   : Dining & Entertaining
title   : You Are Special Today Red Plate [With Red Pen]
cleaned : special today red plate red pen
features: {'Color': 'red'}

cat_3   : Candles & Holders
title   : Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy
cleaned : vicks inhaler relief cold sinus nasal congestion allergy
features: {}

cat_3   : Dining & Entertaining
title   : Artistic Churchware Communion Cup Filler: RW525
cleaned : artistic churchware communion cup filler rw525
features: {'Product_Type': 'cup'}

cat_3   : Bathroom Accessories
title   : 4 BARS! Mysore Sandal Soap 70grams FAST SHIPPING
cleaned : 4 bar mysore sandal soap 70grams fast shipping
features: {}

cat_3   : Home Fragrance
title   : AROGYA VATI (40gm) by popeye seller
cleaned : arogya vati 40gm popeye seller
features: {'Capacity_Volume': '40gm'}



## 3. End-to-end driver

`run_feature_extraction` chains every stage. Pass paths instead of pre-loaded objects to verify the JSON-loading branch works too.

In [ ]:
# Edit this list to add or remove text columns to extract from.
# `list_columns` is the subset whose raw values are stringified lists.
text_columns = ['title', 'description', 'feature']
list_columns = ['description', 'feature']

df_result = run_feature_extraction(
    df_items,
    text_columns=text_columns,
    master_metadata=str(DATA_DIR / 'master_metadata.json'),
    global_filters=str(DATA_DIR / 'global_filters.json'),
    list_columns=list_columns,
    priority=text_columns,   # highest-to-lowest; defaults to text_columns order
)

print(f'Result shape: {df_result.shape}')
print(f'\nNew columns:')
expected = (
    [f'cat_{i+1}' for i in range(6)]
    + [f'{c}_cleaned' for c in text_columns]
    + [f'extracted_features_{c}' for c in text_columns]
    + ['extracted_features']
)
for col in expected:
    print(f'  {col:35s} present: {col in df_result.columns}')

In [22]:
# Per-source vs combined coverage
from collections import Counter

total = len(df_result)
for src in ['title', 'description', 'feature']:
    n = (df_result[f'extracted_features_{src}'].apply(len) > 0).sum()
    print(f'  {src:12s}: {n:,} / {total:,} ({n/total*100:.1f}%)')

n_combined = (df_result['extracted_features'].apply(len) > 0).sum()
print(f'  combined    : {n_combined:,} / {total:,} ({n_combined/total*100:.1f}%)')

field_counts = Counter()
for d in df_result['extracted_features']:
    field_counts.update(d.keys())
print('\nTop fields extracted (combined):')
for field, count in field_counts.most_common(10):
    print(f'  {field:20s} {count:,} ({count/total*100:.1f}%)')

  title       : 1,048,500 / 1,134,566 (92.4%)
  description : 915,857 / 1,134,566 (80.7%)
  feature     : 876,231 / 1,134,566 (77.2%)
  combined    : 1,112,733 / 1,134,566 (98.1%)

Top fields extracted (combined):
  Product_Type         838,262 (73.9%)
  Material             761,739 (67.1%)
  Features             638,614 (56.3%)
  Dimensions           591,890 (52.2%)
  Color                530,245 (46.7%)
  Piece_Count          251,125 (22.1%)
  Theme                181,559 (16.0%)
  Brand                152,854 (13.5%)
  Capacity_Volume      124,755 (11.0%)
  Size                 81,941 (7.2%)


In [23]:
# Inspect a few fully-processed rows
view_cols = ['asin', 'cat_3', 'title_cleaned', 'extracted_features']
df_result[df_result['extracted_features'].apply(len) > 0][view_cols].head(5)

,asin,cat_3,title_cleaned,extracted_features
0,0001487795,Dining & Entertaining,special today red plate red pen,{'Color': 'red'}
2,0006564224,Dining & Entertaining,artistic churchware communion cup filler rw525,"{'Product_Type': 'cup', 'Capacity_Volume': '16 oz'}"
3,0009046461,Bathroom Accessories,4 bar mysore sandal soap 70grams fast shipping,{'Features': 'natural'}
4,0234937912,Home Fragrance,arogya vati 40gm popeye seller,"{'Features': 'natural', 'Capacity_Volume': '40gm'}"
5,0250459655,Posters & Prints,nikola tesla photo nikola tesla quote poster print 12 inch x 18 inch rolled,"{'Features': 'framed', 'Dimensions': '12 inch', 'Product_Type': 'poster prin..."


## 4. Expand to per-field columns

Turn the merged `extracted_features` dict into typed columns: one column per field (`Color`, `Material`, `Dimensions`, …), `<field>_numeric` + `<field>_unit` for capacity/weight/etc., `dimension_1/2/3/_unit` for dimensions, and standardized units via `UNIT_MAP`. This is what produces the table you saw in `create_features.ipynb`.

In [ ]:
from feature_extraction_workflow import expand_features

df_expanded = expand_features(df_result)

print(f'Expanded shape: {df_expanded.shape}')

# Show numeric/unit columns produced
numeric_cols = [c for c in df_expanded.columns if c.endswith('_numeric')]
unit_cols = [c for c in df_expanded.columns if c.endswith('_unit')]
dim_cols = [c for c in df_expanded.columns if c.startswith('dimension_')]

print(f'\nNumeric columns ({len(numeric_cols)}):')
for c in numeric_cols:
    n = df_expanded[c].notna().sum()
    print(f'  {c:30s} {n:,} non-null')

print(f'\nUnit columns ({len(unit_cols)}):')
for c in unit_cols:
    units = df_expanded[c].dropna().unique()
    print(f'  {c:30s} units: {sorted(units)[:10]}')

print(f'\nDimension columns:')
for c in dim_cols:
    n = df_expanded[c].notna().sum()
    print(f'  {c:20s} {n:,} non-null')

In [ ]:
# Spot-check parsing: rows where Capacity_Volume / Dimensions were populated
view_cols = ['asin', 'cat_3', 'Capacity_Volume', 'capacity_volume_numeric', 'capacity_volume_unit']
view_cols = [c for c in view_cols if c in df_expanded.columns]
df_expanded[df_expanded.get('capacity_volume_numeric', pd.Series(dtype=float)).notna()][view_cols].head(5)

In [ ]:
view_cols = ['asin', 'cat_3', 'Dimensions', 'dimension_1', 'dimension_2', 'dimension_3', 'dimension_unit']
view_cols = [c for c in view_cols if c in df_expanded.columns]
df_expanded[df_expanded.get('dimension_1', pd.Series(dtype=float)).notna()][view_cols].head(5)

## 5. Spot-check: title-only run

Confirms the pipeline works with a single text column too.

In [24]:
df_title_only = run_feature_extraction(
    df_items,
    text_columns=['title'],
    master_metadata=master_metadata,   # pass the loaded dict directly
    global_filters=global_filters,
)
n = (df_title_only['extracted_features'].apply(len) > 0).sum()
print(f'Title-only coverage: {n:,} / {len(df_title_only):,} ({n/len(df_title_only)*100:.1f}%)')
df_title_only[['asin', 'cat_3', 'title_cleaned', 'extracted_features']].head(3)

Title-only coverage: 1,048,500 / 1,134,566 (92.4%)


,asin,cat_3,title_cleaned,extracted_features
0,0001487795,Dining & Entertaining,special today red plate red pen,{'Color': 'red'}
1,0002020300,Candles & Holders,vicks inhaler relief cold sinus nasal congestion allergy,{}
2,0006564224,Dining & Entertaining,artistic churchware communion cup filler rw525,{'Product_Type': 'cup'}
